# Carbon Deposition Prediction in Catalytic Methane Pyrolysis

This notebook compares three models for predicting deposited carbon mass:

- **Classic-FNN:** data-fitting neural network.
- **PINN-M1:** physics-informed model with an Arrhenius/power-law carbon formation rate (CFR).
- **PINN-M2:** physics-informed model using measured CFR as the derivative constraint.

The workflow separates training, validation, and test data. The validation set is used for early stopping and model selection; the test set is reserved for final evaluation.

**Important:** This cleaned notebook has not been run against the research dataset in this environment. Confirm the CSV schema and rerun all cells before using or reporting results.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
CSV_PATH = Path("/content/Carbon___Mass___Deposition___Data.csv")  # Change if needed
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["T_C", "Fo_CH4_m3_per_h", "X_CH4_conv_pct", "P_input_W", "TOS_h"]
MASS_COL = "M_C_kg"
CFR_COL = "CFR_kg_per_h"
SEC_COL = "SEC_kWh_per_kgC"

# Retained as the final configuration used in the original notebook.
FINAL_HIDDEN = [128, 64]
FINAL_ACTIVATION = "ReLU"
LEARNING_RATE = 1e-3
MAX_EPOCHS = 10_000
PATIENCE = 1_000
LAMBDA_PHYS = 0.4

R_GAS = 8.314
EPS = 1e-12

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Load and validate the dataset

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {CSV_PATH}. Upload/mount the CSV or update CSV_PATH."
    )

raw_df = pd.read_csv(CSV_PATH)
raw_df.columns = raw_df.columns.str.strip()

required_cols = FEATURE_COLS + [MASS_COL, CFR_COL]
missing_cols = [col for col in required_cols if col not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

for col in required_cols + ([SEC_COL] if SEC_COL in raw_df.columns else []):
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# Model rows require all model features, carbon mass, and CFR.
model_df = raw_df.dropna(subset=required_cols).copy()
model_df = model_df[(model_df[MASS_COL] >= 0) & (model_df[CFR_COL] >= 0)].copy()
if len(model_df) < 10:
    raise ValueError(f"Only {len(model_df)} usable rows found; check the dataset and filters.")

model_df["M_C_g"] = 1000.0 * model_df[MASS_COL]
model_df["CFR_g_per_h"] = 1000.0 * model_df[CFR_COL]

print(f"Rows in original CSV: {len(raw_df)}")
print(f"Rows available for modelling: {len(model_df)}")
display(model_df[FEATURE_COLS + ["M_C_g", "CFR_g_per_h"]].head())

## 3. Exploratory analysis

In [ ]:
# Descriptive statistics and correlation analysis.
summary_cols = [c for c in [SEC_COL, "Y_C_feed_pct", MASS_COL, CFR_COL] if c in model_df.columns]
if summary_cols:
    display(model_df[summary_cols].describe())

corr_cols = [c for c in FEATURE_COLS + [SEC_COL, "Y_C_feed_pct", MASS_COL, CFR_COL] if c in model_df.columns]
display(model_df[corr_cols].corr(numeric_only=True))

# SEC by temperature bins, when SEC is available.
if SEC_COL in model_df.columns:
    sec_df = model_df.dropna(subset=[SEC_COL, "T_C"]).copy()
    bins = [300, 350, 400, 450, 500, 550, 600, 650]
    labels = ["300–350", "350–400", "400–450", "450–500", "500–550", "550–600", "600–650"]
    sec_df["T_bin"] = pd.cut(sec_df["T_C"], bins=bins, labels=labels, include_lowest=True)
    display(sec_df.groupby("T_bin", observed=False)[SEC_COL].agg(["mean", "median", "min", "max", "count"]))

    operational = sec_df[(sec_df[MASS_COL] > 0) & sec_df[SEC_COL].notna()]
    if not operational.empty:
        best_idx = operational[SEC_COL].idxmin()
        cols_to_show = [c for c in ["P_input_W", "Fo_CH4_m3_per_h", "T_C",
                                     "X_CH4_conv_pct", "TOS_h", MASS_COL,
                                     CFR_COL, "Y_C_feed_pct", SEC_COL] if c in operational.columns]
        print("Lowest-SEC row among filtered records (descriptive, not a global optimum):")
        display(operational.loc[[best_idx], cols_to_show])

## 4. Split data and scale features

In [ ]:
X = model_df[FEATURE_COLS].to_numpy(dtype=np.float32)
y_mass = model_df[["M_C_g"]].to_numpy(dtype=np.float32)
y_cfr = model_df[["CFR_g_per_h"]].to_numpy(dtype=np.float32)

# 70% train, 15% validation, 15% test.
X_train_raw, X_temp_raw, y_train, y_temp, cfr_train, cfr_temp = train_test_split(
    X, y_mass, y_cfr, test_size=0.30, random_state=SEED
)
X_val_raw, X_test_raw, y_val, y_test, cfr_val, cfr_test = train_test_split(
    X_temp_raw, y_temp, cfr_temp, test_size=0.50, random_state=SEED
)

# Fit scaler on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val_scaled = scaler.transform(X_val_raw).astype(np.float32)
X_test_scaled = scaler.transform(X_test_raw).astype(np.float32)

TIME_IDX = FEATURE_COLS.index("TOS_h")
SIGMA_T = float(scaler.scale_[TIME_IDX])
if not np.isfinite(SIGMA_T) or SIGMA_T <= 0:
    raise ValueError("TOS_h has zero or invalid training-set scale; cannot compute dM/dt.")

def tensor(array, requires_grad=False):
    return torch.tensor(array, dtype=torch.float32, device=DEVICE).requires_grad_(requires_grad)

X_train_t = tensor(X_train_scaled)
X_val_t = tensor(X_val_scaled)
X_test_t = tensor(X_test_scaled)
X_train_raw_t = tensor(X_train_raw)
X_val_raw_t = tensor(X_val_raw)
y_train_t = tensor(y_train)
y_val_t = tensor(y_val)
y_test_t = tensor(y_test)
cfr_train_t = tensor(cfr_train)
cfr_val_t = tensor(cfr_val)
cfr_test_t = tensor(cfr_test)

print(f"Train: {len(X_train_raw)} | Validation: {len(X_val_raw)} | Test: {len(X_test_raw)}")
print(f"Time feature index: {TIME_IDX}; scaler scale for TOS_h: {SIGMA_T:.6g}")

## 5. Model definitions and physics residuals

All models predict deposited carbon mass in grams and use a non-negative Softplus output. PINN derivatives are taken with respect to the **scaled** time feature and divided by the training-set time scale to recover the derivative per hour.

- **PINN-M1:** \(dM_C/dt\) is constrained by the Arrhenius/power-law CFR expression in the original notebook.
- **PINN-M2:** \(dM_C/dt\) is constrained by measured CFR (converted from kg/h to g/h).

The empirical form and units of the PINN-M1 expression should be checked against the study's documented model definition; this notebook preserves the form used in the supplied code.

In [ ]:
ACTIVATIONS = {
    "ReLU": nn.ReLU,
    "GELU": nn.GELU,
    "Tanh": nn.Tanh,
}

def make_mlp(input_dim, hidden, activation, positive_output=True):
    if activation not in ACTIVATIONS:
        raise ValueError(f"Unsupported activation {activation!r}. Choose from {list(ACTIVATIONS)}.")
    layers = []
    previous = input_dim
    for width in hidden:
        if not isinstance(width, int) or width <= 0:
            raise ValueError(f"Hidden-layer widths must be positive integers; got {width!r}.")
        layers.extend([nn.Linear(previous, width), ACTIVATIONS[activation]()])
        previous = width
    layers.append(nn.Linear(previous, 1))
    if positive_output:
        layers.append(nn.Softplus())
    return nn.Sequential(*layers)

class ClassicFNN(nn.Module):
    def __init__(self, input_dim, hidden, activation):
        super().__init__()
        self.net = make_mlp(input_dim, hidden, activation, positive_output=False)

    def forward(self, x):
        return self.net(x)

class PINN_M1(nn.Module):
    def __init__(self, input_dim, hidden, activation):
        super().__init__()
        self.net = make_mlp(input_dim, hidden, activation, positive_output=True)
        self.k0_raw = nn.Parameter(torch.tensor(0.0))
        self.Ea_raw = nn.Parameter(torch.tensor(0.5))
        self.beta_raw = nn.Parameter(torch.tensor(0.1))

    def forward(self, x_scaled):
        return self.net(x_scaled)

    def k0(self):
        return torch.nn.functional.softplus(self.k0_raw) + EPS

    def Ea(self):
        return 1e5 * (torch.nn.functional.softplus(self.Ea_raw) + EPS)

    def beta(self):
        return torch.nn.functional.softplus(self.beta_raw)

class PINN_M2(nn.Module):
    def __init__(self, input_dim, hidden, activation):
        super().__init__()
        self.net = make_mlp(input_dim, hidden, activation, positive_output=True)

    def forward(self, x_scaled):
        return self.net(x_scaled)

def time_derivative(model, X_scaled):
    # Keep the derivative graph for training physics losses.
    x = X_scaled.detach().clone().requires_grad_(True)
    mass = model(x)
    gradients = torch.autograd.grad(
        outputs=mass,
        inputs=x,
        grad_outputs=torch.ones_like(mass),
        create_graph=True,
    )[0]
    return gradients[:, TIME_IDX:TIME_IDX + 1] / SIGMA_T

def cfr_physics_m1(X_raw_t, model):
    # Original notebook's empirical Arrhenius/power-law expression.
    T_kelvin = X_raw_t[:, 0:1] + 273.15
    flow = X_raw_t[:, 1:2]
    conversion = X_raw_t[:, 2:3] / 100.0
    power_input = X_raw_t[:, 3:4]
    rate = (
        model.k0()
        * torch.exp(-model.Ea() / (R_GAS * T_kelvin))
        * flow
        * conversion
        * torch.pow(power_input + EPS, model.beta())
    )
    return torch.relu(rate)

def physics_loss_m1(model, X_scaled, X_raw_t):
    dM_dt = time_derivative(model, X_scaled)
    cfr_pred = cfr_physics_m1(X_raw_t, model)
    return torch.mean((dM_dt - cfr_pred) ** 2)

def physics_loss_m2(model, X_scaled, cfr_true_t):
    dM_dt = time_derivative(model, X_scaled)
    return torch.mean((dM_dt - cfr_true_t) ** 2)

## 6. Training utility with validation-based checkpoint selection

In [ ]:
def train_model(model, model_name, X_tr, y_tr, X_va, y_va,
                X_tr_raw=None, X_va_raw=None, cfr_tr=None, cfr_va=None,
                lambda_phys=LAMBDA_PHYS, lr=LEARNING_RATE,
                max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Train one model; select checkpoint using validation objective only."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()
    best_val = float("inf")
    best_state = None
    wait = 0
    train_history, val_history = [], []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        pred = model(X_tr)
        data_loss = mse(pred, y_tr)

        if model_name == "PINN-M1":
            phys_loss = physics_loss_m1(model, X_tr, X_tr_raw)
            total_loss = data_loss + lambda_phys * phys_loss
        elif model_name == "PINN-M2":
            phys_loss = physics_loss_m2(model, X_tr, cfr_tr)
            total_loss = data_loss + lambda_phys * phys_loss
        else:
            total_loss = data_loss

        if not torch.isfinite(total_loss):
            raise FloatingPointError(f"{model_name}: non-finite training loss at epoch {epoch}.")

        total_loss.backward()
        optimizer.step()
        train_history.append(float(total_loss.detach().cpu()))

        model.eval()
        with torch.no_grad():
            val_data_loss = mse(model(X_va), y_va).item()

        if model_name == "PINN-M1":
            # Derivative-based validation needs autograd, so do not wrap in no_grad.
            model.eval()
            val_phys_loss = physics_loss_m1(model, X_va, X_va_raw).item()
            val_objective = val_data_loss + lambda_phys * val_phys_loss
        elif model_name == "PINN-M2":
            model.eval()
            val_phys_loss = physics_loss_m2(model, X_va, cfr_va).item()
            val_objective = val_data_loss + lambda_phys * val_phys_loss
        else:
            val_objective = val_data_loss

        val_history.append(float(val_objective))

        if val_objective < best_val:
            best_val = val_objective
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"{model_name}: early stopping at epoch {epoch + 1}.")
                break

    if best_state is None:
        raise RuntimeError(f"{model_name}: no valid checkpoint was saved.")
    model.load_state_dict(best_state)
    return model, {"train_loss": train_history, "val_loss": val_history,
                   "best_val_objective": best_val, "epochs_run": len(train_history)}

## 7. Fit the three models

The original notebook specified `[128, 64]` hidden-layer widths, ReLU activation, learning rate `1e-3`, physics-loss weight `0.4`, a maximum of 10,000 epochs, and patience of 1,000 epochs. These are retained as the default configuration here; they have not been re-optimised in this cleaned version.

Validation data are used for checkpoint selection. The test set is not used in training or early stopping.

In [ ]:
input_dim = X_train_t.shape[1]

model_fnn = ClassicFNN(input_dim, FINAL_HIDDEN, FINAL_ACTIVATION).to(DEVICE)
model_fnn, hist_fnn = train_model(
    model_fnn, "Classic-FNN", X_train_t, y_train_t, X_val_t, y_val_t
)

model_pinn1 = PINN_M1(input_dim, FINAL_HIDDEN, FINAL_ACTIVATION).to(DEVICE)
model_pinn1, hist_pinn1 = train_model(
    model_pinn1, "PINN-M1", X_train_t, y_train_t, X_val_t, y_val_t,
    X_tr_raw=X_train_raw_t, X_va_raw=X_val_raw_t
)

model_pinn2 = PINN_M2(input_dim, FINAL_HIDDEN, FINAL_ACTIVATION).to(DEVICE)
model_pinn2, hist_pinn2 = train_model(
    model_pinn2, "PINN-M2", X_train_t, y_train_t, X_val_t, y_val_t,
    cfr_tr=cfr_train_t, cfr_va=cfr_val_t
)

## 8. Final test-set evaluation

In [ ]:
def evaluate_on_test(model, model_name):
    model.eval()
    with torch.no_grad():
        pred = model(X_test_t).cpu().numpy().reshape(-1)
    true = y_test.reshape(-1)
    metrics = {
        "Model": model_name,
        "MAE_g": mean_absolute_error(true, pred),
        "MSE_g2": mean_squared_error(true, pred),
        "RMSE_g": float(np.sqrt(mean_squared_error(true, pred))),
        "R2": r2_score(true, pred),
    }
    history = {
        "y_true": true,
        "y_pred": pred,
    }
    return metrics, history

metrics_fnn, pred_fnn = evaluate_on_test(model_fnn, "Classic-FNN")
metrics_pinn1, pred_pinn1 = evaluate_on_test(model_pinn1, "PINN-M1")
metrics_pinn2, pred_pinn2 = evaluate_on_test(model_pinn2, "PINN-M2")

metrics_df = pd.DataFrame([metrics_fnn, metrics_pinn1, metrics_pinn2])
display(metrics_df)
metrics_df.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)

for name, result in [
    ("Classic-FNN", pred_fnn),
    ("PINN-M1", pred_pinn1),
    ("PINN-M2", pred_pinn2),
]:
    print(f"\n{name} — first 10 test predictions (true g | predicted g)")
    for true_value, predicted_value in list(zip(result["y_true"], result["y_pred"]))[:10]:
        print(f"{true_value:.3f} | {predicted_value:.3f}")

## 9. Comparison plots

In [ ]:
def plot_model_row(ax_loss, ax_pred, history, predictions, model_name):
    ax_loss.plot(history["train_loss"], label="Training objective")
    ax_loss.plot(history["val_loss"], label="Validation objective")
    ax_loss.set_title(f"{model_name}: loss history")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.grid(True, alpha=0.3)
    ax_loss.legend()

    y_true = predictions["y_true"]
    y_pred = predictions["y_pred"]
    ax_pred.scatter(y_true, y_pred, s=20, alpha=0.75)
    lower = float(min(np.min(y_true), np.min(y_pred)))
    upper = float(max(np.max(y_true), np.max(y_pred)))
    ax_pred.plot([lower, upper], [lower, upper], linestyle="--", label="Ideal agreement")
    ax_pred.set_title(
        f"{model_name}: test predictions\n"
        f"MAE = {mean_absolute_error(y_true, y_pred):.3f} g; "
        f"R² = {r2_score(y_true, y_pred):.3f}"
    )
    ax_pred.set_xlabel("Measured deposited carbon (g)")
    ax_pred.set_ylabel("Predicted deposited carbon (g)")
    ax_pred.grid(True, alpha=0.3)
    ax_pred.legend()

fig, axes = plt.subplots(3, 2, figsize=(14, 15))
plot_model_row(axes[0, 0], axes[0, 1], hist_fnn, pred_fnn, "Classic-FNN")
plot_model_row(axes[1, 0], axes[1, 1], hist_pinn1, pred_pinn1, "PINN-M1")
plot_model_row(axes[2, 0], axes[2, 1], hist_pinn2, pred_pinn2, "PINN-M2")
fig.suptitle("Carbon-deposition prediction model comparison", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plot_path = OUTPUT_DIR / "All_Models_Comparison_Plots.png"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved plot to: {plot_path}")

## 10. Save models and preprocessing metadata

The scaler is required to transform future input rows in the same way as the training data. Save it with the model weights and the configuration used.

In [ ]:
import joblib
import json

torch.save(model_fnn.state_dict(), OUTPUT_DIR / "classic_fnn_state_dict.pt")
torch.save(model_pinn1.state_dict(), OUTPUT_DIR / "pinn_m1_state_dict.pt")
torch.save(model_pinn2.state_dict(), OUTPUT_DIR / "pinn_m2_state_dict.pt")
joblib.dump(scaler, OUTPUT_DIR / "feature_scaler.joblib")

metadata = {
    "seed": SEED,
    "feature_columns": FEATURE_COLS,
    "target_mass_column": "M_C_g",
    "cfr_column": "CFR_g_per_h",
    "hidden_layers": FINAL_HIDDEN,
    "activation": FINAL_ACTIVATION,
    "learning_rate": LEARNING_RATE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "lambda_phys": LAMBDA_PHYS,
    "time_feature_index": TIME_IDX,
    "time_scale": SIGMA_T,
    "note": "Record dataset version/hash and package versions before publication."
}
with open(OUTPUT_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Artifacts saved under: {OUTPUT_DIR.resolve()}")

## 11. Interpretation and limitations

- The model comparison is only as reliable as the input data, split strategy, and physics formulation.
- PINN-M1 uses the supplied Arrhenius/power-law form. Confirm its dimensional consistency and parameter interpretation against the study's equations before scientific reporting.
- The physics-loss weight is not automatically dimensionless; because the data and physics residuals may have different scales, assess loss magnitudes and perform a sensitivity analysis before interpreting the value of `LAMBDA_PHYS`.
- A random row split may place observations from the same experimental condition or time series in both train and test sets. If rows are repeated measurements or grouped by experiment, consider a group-aware split.
- The notebook does not claim a best architecture: the original shared architecture search was removed because it did not consistently instantiate the architectures it reported and its PINN-M2 CFR input was mis-specified. If architecture selection is required, add a corrected search that uses validation data only and records the exact configuration instantiated.
- The test metrics generated by this cleaned workflow are new results and must be recomputed. Do not reuse results from the original notebook without checking how they were produced.